In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# This interactive notebook visualizes ALL 2N poles of the function
#
#       H(s)H(-s)
#
# associated with a continuous-time Butterworth low-pass filter.
#
# Two parameters can be varied:
#
#       N     : filter order
#       ωc    : cutoff angular frequency
#
# The 2N poles lie uniformly on the Butterworth circle of radius ωc.
#
# Only the N poles located in the left half-plane are used in the actual
# stable Butterworth transfer function H(s).
#
# Pole representation:
#
#       Filled red circles : poles used in H(s)
#       Open red circles   : poles rejected because Re{p} > 0
#
# For odd N:
#
#       p_m = ωc exp(jπm/N),       m = 0,1,...,2N-1
#
# For even N:
#
#       p_m = ωc exp[jπ(2m+1)/(2N)],   m = 0,1,...,2N-1
#
# The angular separation between adjacent poles is always
#
#       Δθ = π/N.
#
# For odd N there is one negative real pole in the stable transfer function.
#
# For even N there is no real pole; all stable poles occur in complex
# conjugate pairs.
#
# The table on the right lists all 2N poles and identifies each one as
# USED or REJECTED.
# ==============================================================================

# ==============================================================================
# DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:8px 10px;
    margin:0px 0px 8px 0px;
    font-size:13px;
    line-height:1.45;
    background-color:#f7fbff;
    width:960px;
    max-width:960px;
    box-sizing:border-box;
">
<b>Purpose:</b>
Visualize all 2N poles of H(s)H(-s) for a Butterworth low-pass filter and identify the N poles that form the stable transfer function H(s).
<br>
<b>Interpretation:</b>
All poles lie uniformly on the Butterworth circle of radius ω<sub>c</sub>. Filled red circles correspond to poles in the left half-plane and are used in H(s), whereas open red circles correspond to poles in the right half-plane and are rejected. Changing N modifies the angular distribution, while changing ω<sub>c</sub> changes the circle radius and therefore the radial position of all poles.
</div>
""", layout=Layout(width='970px', max_width='970px'))

# ==============================================================================
# CONTROLS
# ==============================================================================

slider_layout = Layout(width='270px')
style_opts = {'description_width':'80px'}

order_slider = IntSlider(min=1, max=10, step=1, value=5, description='Order N:', continuous_update=True, style=style_opts, layout=slider_layout)

wc_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=5.0, description='ωc:', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)

parameter_title = HTML("""
<div style="
    font-size:14px;
    font-weight:bold;
    margin-top:3px;
    margin-bottom:5px;
">
Filter Parameters:
</div>
""")

info_html = HTML(layout=Layout(width='330px', max_width='330px'))

pole_table = HTML(layout=Layout(width='440px', max_width='440px'))

# ==============================================================================
# FIGURE
# ==============================================================================

fig, ax = plt.subplots(figsize=(6.5, 6.5))

theta_circle = np.linspace(0.0, 2.0 * np.pi, 800)

circle_line, = ax.plot([], [], 'r--', linewidth=1.2, alpha=0.6, label='Butterworth circle')

used_scatter = ax.scatter([], [], s=95, marker='o', facecolors='red', edgecolors='red', linewidths=1.5, label='Used poles')

rejected_scatter = ax.scatter([], [], s=95, marker='o', facecolors='white', edgecolors='red', linewidths=1.8, label='Rejected poles')

ax.axhline(0.0, color='black', linewidth=0.9)
ax.axvline(0.0, color='black', linewidth=0.9)

ax.set_xlabel('Re{s}', fontsize=11)
ax.set_ylabel('Im{s}', fontsize=11)

ax.set_title('Butterworth Poles of H(s)H(-s)', fontsize=13, fontweight='bold', pad=8)

ax.grid(True, linestyle=':', alpha=0.35)

ax.set_aspect('equal', adjustable='box')

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10), ncol=3, fontsize=8)

# Fixed axes so pole motion remains visually meaningful
axis_limit = 5.5

ax.set_xlim(-axis_limit, axis_limit)
ax.set_ylim(-axis_limit, axis_limit)

ax.set_xticks(np.arange(-5, 6, 1))
ax.set_yticks(np.arange(-5, 6, 1))

fig.subplots_adjust(left=0.12, right=0.96, bottom=0.17, top=0.92)

fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.resizable = False

fig.canvas.layout.width = '650px'
fig.canvas.layout.height = '650px'

# ==============================================================================
# UPDATE FUNCTION
# ==============================================================================

def update_butterworth_poles(change=None):

    N = order_slider.value
    wc = wc_slider.value

    # --------------------------------------------------------------------------
    # Compute ALL 2N poles of H(s)H(-s)
    # --------------------------------------------------------------------------

    m = np.arange(0, 2 * N)

    if N % 2 == 1:
        angles = np.pi * m / N
    else:
        angles = np.pi * (2 * m + 1) / (2 * N)

    poles = wc * np.exp(1j * angles)

    # --------------------------------------------------------------------------
    # Stable and rejected poles
    # --------------------------------------------------------------------------

    tolerance = 1e-12

    used_mask = np.real(poles) < -tolerance
    rejected_mask = np.real(poles) > tolerance

    # --------------------------------------------------------------------------
    # Update Butterworth circle
    # --------------------------------------------------------------------------

    circle_x = wc * np.cos(theta_circle)
    circle_y = wc * np.sin(theta_circle)

    circle_line.set_data(circle_x, circle_y)

    # --------------------------------------------------------------------------
    # Update used poles
    # --------------------------------------------------------------------------

    used_poles = poles[used_mask]

    if len(used_poles) > 0:
        used_offsets = np.column_stack((np.real(used_poles), np.imag(used_poles)))
    else:
        used_offsets = np.empty((0, 2))

    used_scatter.set_offsets(used_offsets)

    # --------------------------------------------------------------------------
    # Update rejected poles
    # --------------------------------------------------------------------------

    rejected_poles = poles[rejected_mask]

    if len(rejected_poles) > 0:
        rejected_offsets = np.column_stack((np.real(rejected_poles), np.imag(rejected_poles)))
    else:
        rejected_offsets = np.empty((0, 2))

    rejected_scatter.set_offsets(rejected_offsets)

    # --------------------------------------------------------------------------
    # Pole table
    # --------------------------------------------------------------------------

    rows = ""

    for k, p in enumerate(poles):

        if np.real(p) < -tolerance:

            status = "USED"
            status_style = """
                color:#0066cc;
                background:#eef6ff;
                border:1px solid #9bc8f5;
            """

        elif np.real(p) > tolerance:

            status = "REJECTED"
            status_style = """
                color:#cc0000;
                background:#fff1f1;
                border:1px solid #efaaaa;
            """

        else:

            status = "BOUNDARY"
            status_style = """
                color:#666666;
                background:#f2f2f2;
                border:1px solid #cccccc;
            """

        rows += f"""
        <tr style="border-bottom:1px solid #eeeeee;">
            <td style="
                padding:5px 8px;
                text-align:center;
                font-family:'Times New Roman',serif;
                font-size:17px;
                font-style:italic;
                white-space:nowrap;
            ">
                p<sub>{k}</sub>
            </td>

            <td style="
                padding:5px 10px;
                font-family:'Times New Roman',serif;
                font-size:16px;
                white-space:nowrap;
            ">
                {p.real:+.6f}
                <span style="font-style:italic;">{p.imag:+.6f}j</span>
            </td>

            <td style="
                padding:5px 8px;
                text-align:center;
            ">
                <span style="
                    {status_style}
                    padding:2px 7px;
                    border-radius:10px;
                    font-size:10px;
                    font-weight:bold;
                    letter-spacing:0.3px;
                    white-space:nowrap;
                ">
                    {status}
                </span>
            </td>
        </tr>
        """

    pole_table.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px;
        background:white;
        width:430px;
        max-height:610px;
        overflow-y:auto;
        box-sizing:border-box;
        font-size:12px;
    ">

    <div style="
        font-family:'Times New Roman',serif;
        font-size:18px;
        font-weight:bold;
        margin-bottom:7px;
        text-align:center;
    ">
        Pole Values
    </div>

    <table style="
        width:100%;
        border-collapse:collapse;
    ">

        <tr style="border-bottom:1px solid #bbbbbb;">
            <th style="padding:5px;">Pole</th>
            <th style="padding:5px;">Complex Value</th>
            <th style="padding:5px;">Status</th>
        </tr>

        {rows}

    </table>

    </div>
    """

    # --------------------------------------------------------------------------
    # Information panel
    # --------------------------------------------------------------------------

    delta_theta = np.pi / N
    delta_theta_deg = np.rad2deg(delta_theta)

    if N % 2 == 1:

        parity_text = "Odd order"
        real_pole_text = f"One stable real pole at s = {-wc:.4f}"

    else:

        parity_text = "Even order"
        real_pole_text = "No real pole"

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px 10px;
        margin-top:8px;
        font-size:12px;
        line-height:1.65;
        background:white;
        width:325px;
        box-sizing:border-box;
    ">

    <div>
        <b>Filter:</b>
        <span style="color:#0066cc;">Butterworth low-pass</span>
    </div>

    <div>
        <b>Order N:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Cutoff frequency:</b>
        <span style="color:#0066cc;">ωc = {wc:.2f} rad/s</span>
    </div>

    <div>
        <b>Total poles:</b>
        <span style="color:#0066cc;">{2 * N}</span>
    </div>

    <div>
        <b>Used poles:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Rejected poles:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Angular spacing:</b>
        <span style="color:#0066cc;">π/N = {delta_theta:.4f} rad = {delta_theta_deg:.2f}°</span>
    </div>

    <div>
        <b>Order type:</b>
        <span style="color:#0066cc;">{parity_text}</span>
    </div>

    <div>
        <b>Real pole:</b>
        <span style="color:#0066cc;">{real_pole_text}</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Observation:</b><br>
        All 2N poles lie uniformly on the Butterworth circle. Only the N poles
        in the left half-plane are retained to construct the stable transfer
        function H(s).
    </div>

    </div>
    """

    # --------------------------------------------------------------------------
    # Redraw
    # --------------------------------------------------------------------------

    fig.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

order_slider.observe(update_butterworth_poles, names='value')
wc_slider.observe(update_butterworth_poles, names='value')

# ==============================================================================
# LAYOUT
# ==============================================================================

controls = VBox([parameter_title, order_slider, wc_slider, info_html], layout=Layout(width='340px', min_width='340px', max_width='340px', flex='0 0 340px', align_items='flex-start'))

main_row = HBox([controls, fig.canvas, pole_table], layout=Layout(width='1450px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# INITIALIZE
# ==============================================================================

update_butterworth_poles()

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)
display(main_row)